In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier

In [16]:
train_users = pd.read_csv("train_users.csv")
test_users = pd.read_csv("test_users.csv")
news_df = pd.read_csv("news_articles.csv")

print("Train shape:", train_users.shape)
print("Test shape:", test_users.shape)

Train shape: (2000, 33)
Test shape: (2000, 32)


In [17]:
train_users = train_users.drop(columns=["user_id"], errors="ignore")
test_users = test_users.drop(columns=["user_id"], errors="ignore")

for col in train_users.columns:
    if col != "label":
        train_users[col] = pd.to_numeric(train_users[col], errors="ignore")

for col in test_users.columns:
    test_users[col] = pd.to_numeric(test_users[col], errors="ignore")

/tmp/ipython-input-938086117.py:6: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  train_users[col] = pd.to_numeric(train_users[col], errors="ignore")
/tmp/ipython-input-938086117.py:9: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  test_users[col] = pd.to_numeric(test_users[col], errors="ignore")


In [18]:
X = train_users.drop(columns=["label"])
y = train_users["label"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [19]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [20]:
categorical_cols = X_train.select_dtypes(include="object").columns.tolist()

print("Categorical columns:", categorical_cols)

cat_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

if len(categorical_cols) > 0:
    X_train[categorical_cols] = cat_encoder.fit_transform(
        X_train[categorical_cols]
    )

    X_val[categorical_cols] = cat_encoder.transform(
        X_val[categorical_cols]
    )

Categorical columns: ['browser_version', 'region_code']


In [21]:
print("Object columns left:",
      X_train.select_dtypes(include="object").columns)

Object columns left: Index([], dtype='object')


In [22]:
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


In [23]:
val_pred = model.predict(X_val)

print("\nValidation Accuracy:",
      accuracy_score(y_val, val_pred))

print("\nValidation Classification Report:\n")
print(classification_report(
    y_val,
    val_pred,
    target_names=label_encoder.classes_
))


Validation Accuracy: 0.915

Validation Classification Report:

              precision    recall  f1-score   support

      user_1       0.89      0.89      0.89       142
      user_2       0.98      0.89      0.94       142
      user_3       0.88      0.97      0.92       116

    accuracy                           0.92       400
   macro avg       0.92      0.92      0.92       400
weighted avg       0.92      0.92      0.92       400



In [24]:
X_test = test_users.copy()

if len(categorical_cols) > 0:
    X_test[categorical_cols] = cat_encoder.transform(
        X_test[categorical_cols]
    )

test_predictions = model.predict(X_test)

decoded_predictions = label_encoder.inverse_transform(
    test_predictions
)

print("\nSample Predictions:")
print(decoded_predictions[:20])


Sample Predictions:
['user_2' 'user_1' 'user_1' 'user_1' 'user_1' 'user_3' 'user_3' 'user_1'
 'user_3' 'user_3' 'user_2' 'user_2' 'user_3' 'user_1' 'user_3' 'user_2'
 'user_3' 'user_1' 'user_3' 'user_2']
